# Level 3 final test: CNNs compared with Ridge and MLP

Same protocol as the Ridge and MLP branches:

1. The two CNNs were chosen on validation scores (`run_levels.py`, level 3, fold 5): `model_10_5` is the best 4–10 step CNN, `model_16_5` the best 12–16 step CNN.
2. Both were retrained on all 5 folds with fixed settings (`final_train_cnn.py`, 25 epochs).
3. Each is evaluated **once** here on the level-3 test group (120 simulations, 1,800 maps). This notebook does no training, no fitting and no model selection.

Metrics are RMSE and MAE for $\Omega_m$ and $\sigma_8$ in original units, per map and per simulation (the 15 maps of a simulation averaged first). The baseline always predicts the training-label mean.

Ridge and MLP test results are read straight from the `ridge-regression` and `mlp` branches, so run `git fetch origin` before this notebook. Nothing from those branches is copied into this one.

## 0. Setup

In [ ]:
from pathlib import Path
import csv
import io
import json
import os
import subprocess
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "final_train_cnn.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook inside the cosmoreg repo.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt

from final_train_cnn import LEVEL, FINAL_MODELS, final_path, log_maps
from train_single import get_device, get_model_class

OUT_DIR = PROJECT_ROOT / "cnn_test"
OUT_DIR.mkdir(exist_ok=True)
TARGETS = ["Omega_m", "sigma_8"]
TARGET_TEX = [r"$\Omega_m$", r"$\sigma_8$"]

MODELS = ["ridge", "mlp", "model_10_5", "model_16_5"]
LABELS = {"ridge": "Ridge", "mlp": "MLP",
          "model_10_5": "CNN, 10 steps\n(model_10_5)", "model_16_5": "CNN, 16 steps\n(model_16_5)"}
COLORS = {"ridge": "#2a78d6", "mlp": "#eb6834", "model_10_5": "#1baf7a", "model_16_5": "#eda100"}
MARKERS = {"ridge": "o", "mlp": "s", "model_10_5": "^", "model_16_5": "D"}
BASELINE_COLOR = "#7a7974"

for name in FINAL_MODELS:
    assert final_path(name).exists(), f"{final_path(name)} not found: run final_train_cnn.py first"
print("Project:", PROJECT_ROOT)
print("Final checkpoints found:", [str(final_path(m)) for m in FINAL_MODELS])

## 1. Metric helpers

The same code scores all four models, so the numbers are directly comparable.

In [ ]:
def average_by_simulation(values, sim_ids):
    unique_ids = np.unique(sim_ids)
    return np.stack([values[sim_ids == s].mean(axis=0) for s in unique_ids]), unique_ids


def metrics(pred, true, baseline_mean):
    err = pred - true
    base_err = baseline_mean - true
    return {
        "rmse": np.sqrt(np.mean(err ** 2, axis=0)).tolist(),
        "mae": np.mean(np.abs(err), axis=0).tolist(),
        "baseline_rmse": np.sqrt(np.mean(base_err ** 2, axis=0)).tolist(),
        "baseline_mae": np.mean(np.abs(base_err), axis=0).tolist(),
    }


def read_branch_file(branch, path):
    # File contents from origin/<branch>, without checking the branch out.
    try:
        return subprocess.run(["git", "show", f"origin/{branch}:{path}"], check=True,
                              capture_output=True, cwd=PROJECT_ROOT).stdout
    except (subprocess.CalledProcessError, FileNotFoundError):
        raise FileNotFoundError(f"{path} not found on origin/{branch}. Run `git fetch origin` first.")

## 2. Load the test group and evaluate the two CNNs

Each final checkpoint carries its own input normalisation and training-label mean, so nothing is fitted on test data. The cell also checks that no test simulation was used in training.

In [ ]:
leaf = PROJECT_ROOT / "maps2d" / f"level{LEVEL}" / "test"
X_test_log = log_maps(leaf)
y_test = np.load(leaf / "labels.npy").astype(np.float32)
sim_ids = np.load(leaf / "simulation_ids.npy")
sim_true, sim_unique = average_by_simulation(y_test, sim_ids)
_, per_sim = np.unique(sim_ids, return_counts=True)
assert np.all(per_sim == 15), "every test simulation should have 15 maps"
print(f"Level {LEVEL} test: {len(y_test)} maps from {len(sim_unique)} simulations")

device = get_device()
results, baseline_mean = {}, None
for name in FINAL_MODELS:
    saved = torch.load(final_path(name), map_location="cpu", weights_only=False)
    assert saved["model_name"] == name and saved["level"] == LEVEL
    assert not np.intersect1d(sim_ids, saved["training_simulation_ids"]).size, "test simulations were used in training"
    baseline_mean = np.array(saved["target_mean"], dtype=np.float32)

    model = get_model_class(name)().to(device)
    model.load_state_dict(saved["state_dict"])
    model.eval()
    X = ((X_test_log - saved["input_mean"]) / saved["input_std"]).astype(np.float32)
    with torch.no_grad():
        pred = np.concatenate([model(torch.from_numpy(X[i:i + 64]).to(device)).cpu().numpy()
                               for i in range(0, len(X), 64)])
    assert np.isfinite(pred).all()
    sim_pred, _ = average_by_simulation(pred, sim_ids)
    results[name] = {
        "map_predictions": pred, "simulation_predictions": sim_pred,
        "map_metrics": metrics(pred, y_test, baseline_mean),
        "simulation_metrics": metrics(sim_pred, sim_true, baseline_mean),
        "training_maps": saved["training_maps"],
    }
    print(f"{name}: trained on {saved['training_maps']} maps, evaluated")
del X_test_log, X

## 3. Ridge and MLP test results

Their saved test predictions are scored with the same code as the CNNs. The check line compares these numbers with the ones each branch reported itself; a mismatch would mean the test maps are in a different order.

In [ ]:
mlp = json.loads(read_branch_file("mlp", f"mlp/level{LEVEL}/level{LEVEL}_test_results.json"))
assert np.allclose(mlp["map_targets"], y_test), "MLP test maps are in a different order"
ridge_pred = np.load(io.BytesIO(read_branch_file("ridge-regression", f"results/ridge/test_predictions_level{LEVEL}.npy")))
assert ridge_pred.shape == y_test.shape

for key, pred in (("ridge", ridge_pred), ("mlp", np.array(mlp["map_predictions"]))):
    sim_pred, _ = average_by_simulation(pred, sim_ids)
    results[key] = {
        "map_predictions": pred, "simulation_predictions": sim_pred,
        "map_metrics": metrics(pred, y_test, baseline_mean),
        "simulation_metrics": metrics(sim_pred, sim_true, baseline_mean),
    }

ridge_csv = list(csv.DictReader(io.StringIO(read_branch_file("ridge-regression", "results/ridge/ridge_results.csv").decode())))
ridge_reported = [float(next(r["RMSE"] for r in ridge_csv if int(r["level"]) == LEVEL and r["parameter"] == t)) for t in TARGETS]
diff_ridge = np.max(np.abs(np.array(results["ridge"]["map_metrics"]["rmse"]) - ridge_reported))
diff_mlp = np.max(np.abs(np.array(results["mlp"]["map_metrics"]["rmse"]) - mlp["map_metrics"]["rmse"]))
print(f"RMSE vs. reported by their branches: Ridge diff {diff_ridge:.1e}, MLP diff {diff_mlp:.1e} -> "
      f"{'OK' if max(diff_ridge, diff_mlp) < 1e-4 else 'MISMATCH'}")

## 4. Results table

RMSE reduction = 100 × (1 − model RMSE / baseline RMSE). Higher is better; 0% means no better than always predicting the mean.

In [ ]:
rows = []
for key in MODELS:
    mm, sm = results[key]["map_metrics"], results[key]["simulation_metrics"]
    for t, target in enumerate(TARGETS):
        rows.append({"model": LABELS[key].replace("\n", " "), "target": target,
                     "RMSE": mm["rmse"][t], "MAE": mm["mae"][t],
                     "sim_RMSE": sm["rmse"][t], "sim_MAE": sm["mae"][t],
                     "baseline_RMSE": mm["baseline_rmse"][t], "baseline_MAE": mm["baseline_mae"][t],
                     "RMSE_reduction_%": 100 * (1 - mm["rmse"][t] / mm["baseline_rmse"][t])})

print(f"Level {LEVEL} test, original units (per map unless marked sim)\n")
print(f"{'model':<28} {'target':<8} {'RMSE':>7} {'MAE':>7} {'sim RMSE':>9} {'sim MAE':>8} {'reduction':>10}")
for r in rows:
    print(f"{r['model']:<28} {r['target']:<8} {r['RMSE']:>7.4f} {r['MAE']:>7.4f} {r['sim_RMSE']:>9.4f} "
          f"{r['sim_MAE']:>8.4f} {r['RMSE_reduction_%']:>9.1f}%")
b = results["ridge"]["map_metrics"]
print(f"{'Baseline (training mean)':<28} {'Omega_m':<8} {b['baseline_rmse'][0]:>7.4f} {b['baseline_mae'][0]:>7.4f}")
print(f"{'Baseline (training mean)':<28} {'sigma_8':<8} {b['baseline_rmse'][1]:>7.4f} {b['baseline_mae'][1]:>7.4f}")

with open(OUT_DIR / f"level{LEVEL}_test_comparison.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

report = {"evaluation_group": "test", "level": LEVEL, "target_names": TARGETS,
          "test_maps": int(len(y_test)), "test_simulations": int(len(sim_unique)),
          "baseline_target_mean": baseline_mean.tolist(),
          "models": {name: {"checkpoint": str(final_path(name).as_posix()),
                            "training_maps": results[name]["training_maps"],
                            "map_metrics": results[name]["map_metrics"],
                            "simulation_metrics": results[name]["simulation_metrics"],
                            "map_predictions": results[name]["map_predictions"].tolist(),
                            "simulation_predictions": results[name]["simulation_predictions"].tolist()}
                     for name in FINAL_MODELS},
          "map_targets": y_test.tolist(), "map_simulation_ids": sim_ids.tolist(),
          "simulation_targets": sim_true.tolist(), "simulation_ids": sim_unique.tolist()}
(OUT_DIR / f"level{LEVEL}_test_results.json").write_text(json.dumps(report, indent=2) + "\n")
print("\nSaved:", OUT_DIR / f"level{LEVEL}_test_comparison.csv", "and", OUT_DIR / f"level{LEVEL}_test_results.json")

## 5. RMSE and MAE by model

Per-map test errors. The dashed line is the baseline; a bar below it means the model learned something.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
x = np.arange(len(MODELS))
for row, metric in enumerate(["rmse", "mae"]):
    for t, ax in enumerate(axes[row]):
        values = [results[k]["map_metrics"][metric][t] for k in MODELS]
        base = results["ridge"]["map_metrics"][f"baseline_{metric}"][t]
        top = max(values + [base]) * 1.15
        ax.bar(x, values, width=0.62, color=[COLORS[k] for k in MODELS])
        for xi, v in zip(x, values):
            ax.text(xi, v + top * 0.015, f"{v:.3f}", ha="center", va="bottom", fontsize=9, color="#3a3936")
        ax.axhline(base, linestyle="--", color=BASELINE_COLOR, linewidth=1.3)
        ax.set_xticks(x)
        ax.set_xticklabels([LABELS[k] for k in MODELS], fontsize=9)
        ax.set_ylim(0, top)
        ax.set_ylabel(f"Test {metric.upper()}")
        ax.set_title(f"{metric.upper()} for {TARGET_TEX[t]}\n(dashed line: baseline {base:.3f})", fontsize=11)
        ax.grid(axis="y", color="#e6e5e1", linewidth=0.8)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
fig.suptitle(f"Level {LEVEL} final test, per map ({len(y_test)} maps)", y=1.0)
fig.tight_layout(h_pad=2.5)
fig.savefig(OUT_DIR / f"level{LEVEL}_test_errors.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", OUT_DIR / f"level{LEVEL}_test_errors.png")

## 6. Predicted vs true, per simulation

Each point is one test simulation (its 15 map predictions averaged). Points on the dashed diagonal are perfect predictions; the dotted line is the baseline, which predicts the same value for every simulation. Read across a row to compare the four models on one parameter.

In [ ]:
fig, axes = plt.subplots(2, len(MODELS), figsize=(3.4 * len(MODELS), 8.2))
for t in range(2):
    values = np.concatenate([results[k]["simulation_predictions"][:, t] for k in MODELS] + [sim_true[:, t]])
    margin = 0.06 * np.ptp(values)
    limits = (values.min() - margin, values.max() + margin)
    for col, key in enumerate(MODELS):
        ax = axes[t, col]
        ax.plot(limits, limits, linestyle="--", color=BASELINE_COLOR, linewidth=1)
        ax.axhline(baseline_mean[t], linestyle=":", color=BASELINE_COLOR, linewidth=1.2)
        ax.scatter(sim_true[:, t], results[key]["simulation_predictions"][:, t], s=22, color=COLORS[key],
                   marker=MARKERS[key], edgecolors="white", linewidths=0.6)
        ax.set_xlim(limits)
        ax.set_ylim(limits)
        ax.set_aspect("equal", adjustable="box")
        ax.spines[["top", "right"]].set_visible(False)
        rmse = results[key]["simulation_metrics"]["rmse"][t]
        ax.set_title(f"{LABELS[key].splitlines()[0]}\nsim RMSE {rmse:.3f}", fontsize=10)
        ax.set_xlabel(f"True {TARGET_TEX[t]}")
        if col == 0:
            ax.set_ylabel(f"Predicted {TARGET_TEX[t]}")
fig.suptitle(f"Level {LEVEL} final test: {len(sim_unique)} simulations", y=1.0)
fig.tight_layout(h_pad=2.5)
fig.savefig(OUT_DIR / f"level{LEVEL}_test_predictions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", OUT_DIR / f"level{LEVEL}_test_predictions.png")

## Notes for the report

- All four models are scored with the same metric code on the same 1,800 test maps, and all were trained on all 5 development folds of level 3.
- The inputs differ: Ridge and MLP use the 50-bin power spectrum of each map, the CNNs use the full 256×256 map.
- Each CNN result comes from one training run (seed 42), and the CNNs were chosen on a single validation fold.